In [44]:
from gensim.models import KeyedVectors

import pandas as pd
import numpy as np 

import re
import html 
import string



from Preprocessing import Datacleaner
from visualize import Visualizer

from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [3]:
df = pd.read_csv("IMDB Dataset.csv")
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


Preprocessing


In [4]:
Datacleaner.remove_duplicated(df)

df['review'] = df['review'].apply(Datacleaner.lower_string)

df['review'] = df['review'].apply(Datacleaner.remove_html)

df['review'] = df['review'].apply(Datacleaner.remove_punctuation)

Datacleaner.remove_duplicated(df)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


Tokenization and Stopwords removal


In [ ]:
df['tokenize'] = df['review'].apply(Datacleaner.tokenize_text)

df['stopwords'] = df['tokenize'].apply(Datacleaner.remove_stopwords)

df['clean_text'] = df['stopwords'].apply(lambda x: ' '.join(x) if x else '')


Mapping Of Sentiment

In [10]:
df['sentiment'] = df['sentiment'].map({
    "positive" : 1, 
    "negative" : 0
})

In [20]:
reviews = df['clean_text'].values
labels = df['sentiment'].values

In [ ]:
vocal_size = 20000
tokenize = Tokenizer(num_words=vocal_size, oov_token="")
tokenize.fit_on_texts(reviews)

In [ ]:
sequence = tokenize.texts_to_sequences(reviews)
max_length = 200

padded_sequence = pad_sequences(sequence, maxlen=max_length,padding='post',truncating='post')
X_train,X_test,y_train,y_test = train_test_split(padded_sequence,labels,test_size=0.2,random_state=42)

In [31]:
word2vec_path = r'C:\Users\shiva\gensim-data\word2vec-google-news-300\word2vec-google-news-300.gz'
word2vec = KeyedVectors.load_word2vec_format(word2vec_path,binary=True)

Creating Embedding Matrix

In [ ]:
word_index = tokenize.word_index
embedding_dim = 300
embedding_matrix = np.zeros((vocal_size,embedding_dim))

for word,i in word_index.items():
    if i < vocal_size:
        if word in word2vec:
            embedding_matrix[i] = word2vec[word]

In [39]:
model = Sequential([
    Embedding(input_dim=vocal_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_length,
              trainable=False),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1,activation='sigmoid')
])

In [40]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [41]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 300)          6000000   
                                                                 
 lstm (LSTM)                 (None, 128)               219648    
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 6219777 (23.73 MB)
Trainable params: 219777 (858.50 KB)
Non-trainable params: 6000000 (22.89 MB)
_________________________________________________________________


Train Model

In [43]:
history = model.fit(X_train,y_train,epochs=10,batch_size=128,validation_data=(X_test,y_test))


Epoch 1/10
310/310 [==============================] - 783s 3s/step - loss: 0.6861 - accuracy: 0.5286 - val_loss: 0.6876 - val_accuracy: 0.5180
Epoch 2/10
 69/310 [=====>........................] - ETA: 10:52 - loss: 0.6589 - accuracy: 0.5930

KeyboardInterrupt: 